# QDC 2026 challenge: Estimate a 56-qubit operator Loschmidt echo that nobody can compute

An **operator Loschmidt echo (OLE)** asks how much of an observable survives when a dynamics is run
forward, perturbed, and run back. This challenge uses the 56-qubit instance of
[Barron et al., *Observable Estimation in the Absence of Classical Verification*](https://arxiv.org/abs/2607.25998),
run at half the paper's depth. No classical method can check your numbers. Your job is to produce
the best estimates you can on a real device, with error bars you would stand behind, and to show
evidence that they're right.

How you get there is up to you. Any technique, library, service or analysis is allowed, as long as
the numbers come from your own runs on the event backend within the budget. The QDC 2026 challenges
cheat sheet lists tools that can help at every step.

**What's graded**

| Exercise | What you submit | Points |
|---|---|---:|
| 1. Build the echo | Your circuit-building function | 10 |
| 2. Place and compile for ibm_boston | Your compiled circuit and the qubits it runs on | 10 |
| 3. Best estimate of the echo | Estimates and errors at five points, with your job IDs | 55 |
| 4. Survive a noise change | Estimates at two noise levels, with your job IDs | 18 |
| Questions | Seven multiple-choice answers | 7 |

Exercise 3 is open-ended: its precision credit keeps growing as your errors shrink, so there's
always more to gain. Nobody is expected to reach 100.

**Rules**

- Backend: **ibm_boston**. Budget: at most **600 QPU seconds per complete run** (every job that
  contributes to a submission, including any noise characterization). Across the event, aim to stay
  within about 1,800 seconds in total; this is a guideline, not a graded limit.
- Hardware numbers are self-reported with their job IDs, which the organizers may audit after the
  event. Grade only your own jobs, from runs made during the event.
- Use the 64 inputs defined below for every estimate.

## The instance: OLE-QDC26-56

Qubits carry the physical labels of the published ibm_boston layout; you may move the instance to
other qubits in Exercise 2.

| Item | Definition |
|---|---|
| Graph | 56 qubits and 62 couplers of a heavy-hex patch, in 3 **edge groups** (22, 20 and 20 couplers) that never share a qubit |
| Rotations | $\theta_{\rm fast}=3\pi/16$ on 47 qubits, $\theta_{\rm slow}=0.25$ on 9; $R_Z(\pi/4)$ |
| Scattering | 11 qubits $V_S$ use $\theta_q-\eta$ |
| Depth | $L=3$ layers per block (the paper uses 6) |
| Observable | $O=\prod_{q\in V_O}Z_q$ on 12 qubits |
| Probe | $V_\delta=R_X(2\delta)$ on 35 qubits $V_P$ |

**One layer** $F(\eta)$ runs the three edge groups in order. For each group: on every edge $(a,b)$,
$R_X(\theta_a)$ and $R_Z(\pi/4)$ on $a$, then the same on $b$; then $R_{ZZ}(\pi/2)$ on every edge; then
$R_X(\theta_a)$ and $R_X(\theta_b)$ on every edge again; then a barrier on the instance's qubits. With
$U_0=F(0)^L$ and $U_\eta=F(\eta)^L$, the echo is, in time order, $U_0$, $U_\eta^\dagger$, $V_\delta$, $U_\eta$,
$U_0^\dagger$, with a barrier on the instance's qubits after each block.

**The quantity.** For an input $|x\rangle$, run the echo and measure $\langle O\rangle_x$. The echo signal is

$$S(\eta,\delta)=\frac1{|X|}\sum_{x\in X}s(x)\langle O\rangle_x,\qquad s(x)=(-1)^{\sum_{q\in V_O}x_q}.$$

At $\eta=0$ or $\delta=0$ it is exactly 1. Elsewhere nobody knows its value.

In [ ]:
import json
import sys
from pathlib import Path

OLE_ROOT = Path.cwd().resolve()
if OLE_ROOT.name == "challenge":
    OLE_ROOT = OLE_ROOT.parent
sys.path.insert(0, str(OLE_ROOT))

import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

from utils.instance56 import NUM_DEVICE_QUBITS, SPEC, instance_edges

QUBITS, EDGE_GROUPS = SPEC["qubits"], [[tuple(e) for e in group] for group in SPEC["edge_groups"]]
FAST, SCATTERING = set(SPEC["fast"]), set(SPEC["scattering"])
PROBE, OBSERVABLE = SPEC["probe"], SPEC["observable"]
THETA, RZ_ANGLE, L = SPEC["theta"], SPEC["rz_angle"], 3
INPUTS = np.random.default_rng(2026).integers(0, 2, size=(64, len(QUBITS)))  # INPUTS[x, k] is the bit of QUBITS[k]
ETA_SWEEP = 11 * np.pi / 40
POINTS = [(0.0, 0.3), (ETA_SWEEP, 0.0), (ETA_SWEEP, 0.1), (ETA_SWEEP, 0.2), (ETA_SWEEP, 0.3)]  # Exercise 3
FOLD_POINTS = [(np.pi / 8, 0.3), (ETA_SWEEP, 0.3), (13 * np.pi / 40, 0.3)]                    # Exercise 4
ANSWERS = {}


def signs(inputs: np.ndarray = INPUTS) -> np.ndarray:
    # s(x) for each input row.
    return (-1.0) ** np.asarray(inputs)[:, [QUBITS.index(q) for q in OBSERVABLE]].sum(axis=1)


print(f"{len(QUBITS)} qubits, {len(instance_edges())} couplers, {len(INPUTS)} inputs")

Grading goes through the challenge client, and QDC 2026 is played in teams: answers are only graded
once you have joined your team. Set `TEAM_NAME` below and run the cell before you grade anything.
Every member must type the team name exactly the same way (capitals and spaces count). It can have
letters, digits, spaces, hyphens and underscores, up to 35 characters. Your team's best score on each
exercise counts, so you can resubmit.

In [ ]:
from qc_grader.challenges.qdc_2026 import (
    check_progress, grade_lab_ole_ex1, grade_lab_ole_ex2, grade_lab_ole_ex3, grade_lab_ole_ex4,
    grade_lab_ole_questions, join_team,
)

In [ ]:
TEAM_NAME = ""  # YOUR TEAM NAME, the same for every member

if TEAM_NAME:
    join_team(TEAM_NAME)
else:
    print("Set TEAM_NAME to your team's name and run this cell again: answers are only graded for a team.")

## Exercise 1: Build the echo (10 points)

<div class="alert alert-block alert-success">

✏️ **Your task.** Write `echo(eta, delta)`: the echo above as an unmeasured `QuantumCircuit(NUM_DEVICE_QUBITS)` on the
published qubit labels, accepting floats or `Parameter`s. The grader calls it at two points and
compares each circuit with the paper's, qubit by qubit, up to global phase: barriers, and how you
write each single-qubit rotation, don't matter.

</div>

In [ ]:
def echo(eta, delta) -> QuantumCircuit:
    # Return U0, U_eta^dagger, V_delta, U_eta, U0^dagger, each followed by a barrier on QUBITS.
    # YOUR CODE HERE
    return QuantumCircuit(NUM_DEVICE_QUBITS)

In [ ]:
grade_lab_ole_ex1(echo)

<div class="alert alert-block alert-success">

✏️ **Question (1 point).** At $\eta=0$, which statement about the echo is correct?

- **A.** $S$ falls as $\delta$ grows, because the probe still rotates its 35 qubits.
- **B.** $S=1$ for every $\delta$: the blocks around the probe cancel, and $V_P$ misses $V_O$.
- **C.** $S=\cos^2(2\delta)$, because $U_0$ carries the probe's rotation onto the observable's qubits.
- **D.** $S=0$, because without scattering the observable spreads over every qubit.

</div>

In [ ]:
ANSWERS["eta0"] = ""  # "A", "B", "C" or "D"

## Exercise 2: Place and compile for ibm_boston (10 points)

Choose the 56 physical qubits the instance runs on, and compile a template that prepares any input
and runs the echo, so that one compiled circuit serves every input and point.

<div class="alert alert-block alert-success">

✏️ **Your task.** Set `layout` (the physical qubit of `QUBITS[k]`, for each `k`) and `isa`, a compiled circuit for
ibm_boston on those qubits. `isa` must keep free parameters for the input bits and for `eta` and
`delta`, use only the backend's native gates, and apply its two-qubit gates exactly on the images of
the instance's couplers. Full credit needs two-qubit depth 36 (one layer per edge group: 744 two-qubit
gates); a valid deeper compilation earns half.

</div>

In [ ]:
layout = list(QUBITS)  # YOUR CHOICE
isa = QuantumCircuit(NUM_DEVICE_QUBITS)  # YOUR CODE HERE

In [ ]:
grade_lab_ole_ex2(isa, layout)

<div class="alert alert-block alert-success">

✏️ **Question (1 point).** The echo has 744 $R_{ZZ}(\pi/2)$ gates. A straightforward compilation uses twice as many CZ gates. What's the minimum?

- **A.** One: $R_{ZZ}(\pi/2)$ is a CZ with $R_Z$ rotations on both qubits, up to a phase.
- **B.** Two: one CZ can't make an $R_{ZZ}$ rotation, so the compilation is already minimal.
- **C.** Two, because every entangling rotation other than CZ itself needs two CZ gates.
- **D.** Three, because a general two-qubit gate needs three, and $R_{ZZ}$ is no exception.

</div>

In [ ]:
ANSWERS["cz"] = ""  # "A", "B", "C" or "D"

<div class="alert alert-block alert-success">

✏️ **Question (1 point).** The compiled echo takes about 10 µs, measurement included, and ibm_boston's default repetition delay is 250 µs. What dominates the QPU time of each shot?

- **A.** The circuit's duration, because its 744 CZ gates run in 36 sequential layers.
- **B.** The repetition delay, so QPU time scales with the number of circuits and shots.
- **C.** Measurement, because reading out all 56 qubits takes longer than the gates themselves.
- **D.** Classical processing, because parameters are bound again before every shot.

</div>

In [ ]:
ANSWERS["repdelay"] = ""  # "A", "B", "C" or "D"

## Exercise 3: Best estimate of the echo (55 points)

Estimate $S$ at the five `POINTS`: $\eta=0,\ \delta=0.3$ and the zero-probe control $\eta=11\pi/40,\
\delta=0$, whose exact values are 1, and the $\delta$ sweep at $\eta=11\pi/40$, whose values nobody knows.
Report an estimate and a one-standard-error uncertainty for each, from runs on ibm_boston within
600 QPU seconds.

**How it's scored**

- **30 points** for a complete submission: all five points, finite values, positive errors, ibm_boston,
  and at most 600 QPU seconds over the jobs you list.
- **10 points** for honesty at the two exact-valued points: 5 each when your estimate is within two
  standard errors of 1, and 2.5 when it's within three, scaled by $\min(1,\,0.1/\sigma)$ so that only an
  informative error bar earns full credit. A point reported as exactly 1 (for example, one you
  normalized by) tests nothing and earns none.
- **15 points** for precision: $15\times0.05/(0.05+\bar\sigma)$, where $\bar\sigma$ is your mean error over the
  five points. It approaches 15 as your errors approach zero.

Noise attenuates the raw signal far below 1, so an honest small error bar needs a correction you can
defend. A correction that is precise but biased will show up at the exact-valued points.

<div class="alert alert-block alert-success">

✏️ **Your task.** Fill in `estimates` and `errors` (one entry per point in `POINTS`, in order) and `jobs`, the
runtime jobs of the run they come from: every job, including any characterization runs.

</div>

In [ ]:
estimates = [0.0] * len(POINTS)  # YOUR RESULTS
errors = [0.0] * len(POINTS)
jobs = []                         # your RuntimeJobV2 objects, or their IDs

In [ ]:
if jobs:
    grade_lab_ole_ex3(POINTS, estimates, errors, jobs)
else:
    print("List your jobs in jobs to grade Exercise 3.")

<div class="alert alert-block alert-success">

✏️ **Question (1 point).** At the paper's depth ($L=6$), an earlier test run measured zero-probe controls ($\delta=0$) between −0.002 and 0.017, all within 4.5 standard errors of zero, instead of 1. What follows?

- **A.** The circuit is wrong, because noise can't move a $\delta=0$ control that far from 1.
- **B.** The probe adds the most noise at $\delta=0$, so those points are the least reliable.
- **C.** Readout error causes it, so correcting readout alone would bring all the controls back to 1.
- **D.** Dividing by a control only works if it is resolved well above its error, and here it isn't.

</div>

In [ ]:
ANSWERS["control"] = ""  # "A", "B", "C" or "D"

<div class="alert alert-block alert-success">

✏️ **Question (1 point).** You average 64 per-input estimates into $S$ and combine their standard errors as $\sqrt{\sum_x\sigma_x^2}/64$. Which statement is correct?

- **A.** It's always valid, because it is the standard error of any average of estimates.
- **B.** It should be the mean of the $\sigma_x$, because every input gets the same number of shots.
- **C.** It assumes independent inputs; random choices shared across inputs would make it too small.
- **D.** It should be $\sum_x\sigma_x/64$ instead, because the signs $s(x)$ can cancel between inputs.

</div>

In [ ]:
ANSWERS["errors"] = ""  # "A", "B", "C" or "D"

## Exercise 4: Survive a noise change (18 points)

Without an exact answer, a correction earns trust if it **doesn't change when the noise does**.
Estimate $S$ at the three `FOLD_POINTS` twice: at the device's native noise level, and with the noise
deliberately amplified by a known factor of at least 1.5. Use the same correction method at both
levels.

**How it's scored**

- **6 points** for a complete test: both noise levels, all three points, a noise factor of at least
  1.5, finite values and positive errors, within the budget.
- **4 points** for a decisive test: $4\times\min(1,\,0.03/\sigma_{\max})$, where $\sigma_{\max}$ is your largest error.
- **8 points** if the two levels agree, with $\chi^2=\sum_k(a_k-b_k)^2/(\sigma_{a,k}^2+\sigma_{b,k}^2)$ at most 2 per
  point (4 points if at most 4), scaled by the same $\min(1,\,0.03/\sigma_{\max})$: large errors make any two
  levels agree.

<div class="alert alert-block alert-success">

✏️ **Your task.** Fill in the estimates and errors at both noise levels (one entry per point in `FOLD_POINTS`), the
noise factor you used, and the jobs they come from.

</div>

In [ ]:
native, native_errors = [0.0] * 3, [0.0] * 3        # YOUR RESULTS
amplified, amplified_errors = [0.0] * 3, [0.0] * 3
noise_factor = 1.5
fold_jobs = []

In [ ]:
if fold_jobs:
    grade_lab_ole_ex4(FOLD_POINTS, native, native_errors, amplified, amplified_errors, noise_factor, fold_jobs)
else:
    print("List your jobs in fold_jobs to grade Exercise 4.")

<div class="alert alert-block alert-success">

✏️ **Question (1 point).** On an earlier device, a folded run's zero-probe controls came out *higher* than a separate native-noise run's, even though folding adds two-qubit noise. What's the most likely explanation?

- **A.** Folding replaces noisy gates with cleaner ones, so it can lower the total error.
- **B.** The device drifted: the runs were hours apart, and it was better for the folded one.
- **C.** Rescaling is invalid for this circuit, so controls from two runs can't be compared.
- **D.** Shot noise: the difference is a statistical fluctuation and needs no explanation.

</div>

In [ ]:
ANSWERS["drift"] = ""  # "A", "B", "C" or "D"

<div class="alert alert-block alert-success">

✏️ **Question (1 point).** A **canary** is a circuit with a known exact value, run alongside an experiment in consecutive windows. What does comparing a canary's attenuation with the echo's zero-probe control test?

- **A.** Whether the echo's final mitigated estimate is exact at every one of its points.
- **B.** Whether the error cancellation applied to the echo's points is free of any bias.
- **C.** Whether rescaling the echo by its control is necessary at all.
- **D.** Whether the canary sees noise like the echo's, so its windows can track drift.

</div>

In [ ]:
ANSWERS["canary"] = ""  # "A", "B", "C" or "D"

## Submit your answers

In [ ]:
grade_lab_ole_questions(ANSWERS)
check_progress()

## References

1. S. V. Barron et al., [Observable Estimation in the Absence of Classical Verification](https://arxiv.org/abs/2607.25998), arXiv:2607.25998 (2026).
2. Instance circuits: [Quantum Advantage Tracker](https://github.com/quantum-advantage-tracker/quantum-advantage-tracker.github.io), `operator_loschmidt_echo_56x1488` (Apache-2.0).

© IBM Corp. 2026